INSTALL

In [ ]:
import sys

# make sure we're on the right Python version before installing anything
REQUIRED_VERSION = (3, 11, 9)
actual_version   = sys.version_info[:3]

if actual_version != REQUIRED_VERSION:
    required_str = '.'.join(str(v) for v in REQUIRED_VERSION)
    actual_str = '.'.join(str(v) for v in actual_version)
    raise RuntimeError(
        f"Wrong Python version: expected {required_str}, got {actual_str}. "
        "check that the correct kernel is selected"
        "download from: https://www.python.org/downloads/release/python-3119/"
    )

print(f"Python {'.'.join(str(v) for v in actual_version)} ✓")

%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu --force-reinstall
%pip install pandas numpy Pillow tqdm scikit-learn xgboost

IMPORTS

In [ ]:
# data manipulation
import pandas as pd
import numpy as np

# deep learning - model, layers, transforms, data loading
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset

# image loading
from PIL import Image

# utilities
import os as os
import tqdm as tqdm

# classical ML - models and metrics
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier

PREPROCESSING

In [ ]:
# paths to the raw image folders
train_image_dir = r'Data\task1_data\images\train'
test_image_dir  = r'Data\task1_data\images\test'

# fail safes
if not os.path.exists(train_image_dir):
    raise FileNotFoundError(f"Training directory not found: {train_image_dir}")

if not os.path.exists(test_image_dir):
    raise FileNotFoundError(f"Test directory not found: {test_image_dir}")

SAVE PREPROCESSED IMAGES

In [ ]:
# ResNet requires 224x224 inputs.
# Mean and std values are from ImageNet (the dataset ResNet was trained on).
# Normalising to the same distribution ResNet expects ensures meaningful feature extraction.
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def image_to_resnet(image_dir: str, output_path: str, has_labels: bool = True) -> None:
    image_tensors = []
    label_list = []
    path_list = []

    if has_labels:
        # each subfolder name is a class label
        class_names = sorted(os.listdir(image_dir))
        class_to_idx  = {class_name: idx for idx, class_name in enumerate(class_names)}

        # count total images
        total_images = sum(
            len(os.listdir(os.path.join(image_dir, cls)))
            for cls in class_names
            if os.path.isdir(os.path.join(image_dir, cls))
        )

        num_processed = 0
        for class_name in class_names:
            class_image_dir = os.path.join(image_dir, class_name)

            # skip anything that isn't a folder
            if not os.path.isdir(class_image_dir):
                continue

            for filename in os.listdir(class_image_dir):
                # skip non-image files
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue

                image_path = os.path.join(class_image_dir, filename)
                img = Image.open(image_path).convert("RGB")

                image_tensors.append(transform(img))
                label_list.append(class_to_idx[class_name])
                path_list.append(image_path)

                num_processed += 1
                print(f'{num_processed}/{total_images}', end='\r')

        torch.save({
            'tensors': torch.stack(image_tensors),
            'labels': torch.tensor(label_list),
            'paths': path_list,
            'class_to_idx': class_to_idx
        }, output_path)

    else:
        # test folder is flat so no labels
        image_filenames = [
            f for f in os.listdir(image_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        total_images = len(image_filenames)

        for image_idx, filename in enumerate(image_filenames, start=1):
            image_path = os.path.join(image_dir, filename)
            img = Image.open(image_path).convert("RGB")

            image_tensors.append(transform(img))
            path_list.append(image_path)

            print(f'{image_idx}/{total_images}', end='\r')

        torch.save({
            'tensors': torch.stack(image_tensors),
            'paths': path_list
        }, output_path)

    print(f"\nSaved {len(image_tensors)} images to {output_path}")



In [ ]:
image_to_resnet(train_image_dir, r'Data\task1_data\t1_train.pt', has_labels=True)
image_to_resnet(test_image_dir, r'Data\task1_data\t1_test.pt',  has_labels=False)

RESNET FEATURE EXTRACTION

In [ ]:
# load ResNet50 with pretrained ImageNet weights
resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# strip the final classification head, we only want the feature extractor part.
# this leaves a model that outputs a 2048-d vector per image instead of class scores.
resnet = nn.Sequential(*list(resnet.children())[:-1])
resnet.eval()  # inference only, no gradient tracking needed

def extract_resnet_features(pt_path: str, output_path: str) -> None:
    saved_data = torch.load(pt_path, weights_only=False)
    image_tensors = saved_data['tensors']   # shape: (N, 3, 224, 224)
    total_images = len(image_tensors)
    feature_list = []

    with torch.no_grad():
        for image_idx, image_tensor in enumerate(image_tensors, start=1):
            # ResNet expects a batch dimension: (3,224,224) -> (1,3,224,224)
            feature_vector = resnet(image_tensor.unsqueeze(0))  # output: (1, 2048, 1, 1)

            # remove the batch and spatial dims: (1, 2048, 1, 1) -> (2048,)
            feature_vector = feature_vector.squeeze().numpy()

            feature_list.append(feature_vector)
            print(f'{image_idx}/{total_images}', end='\r')

    # stack all individual feature vectors into one (N, 2048) matrix
    features = np.array(feature_list)

    # merge the feature matrix into the original data dict and save
    output_data = {**saved_data, 'features': features}
    torch.save(output_data, output_path)
    print(f"\nExtracted features shape: {features.shape} -> saved to {output_path}")



In [ ]:
extract_resnet_features(r'Data\task1_data\t1_train.pt', r'Data\task1_data\t1_train_resnet.pt')
extract_resnet_features(r'Data\task1_data\t1_test.pt',  r'Data\task1_data\t1_test_resnet.pt')

CLASSIFICATION MODELS

In [ ]:
"""
Classification model definitions. We try four different approaches:
  1. Linear Classifier  (logistic regression)
  2. SVM                (support vector machine)
  3. XGBoost            (gradient boosted trees)
  4. MLP                (multi-layer perceptron, a small neural network)

All train_* functions share the same interface:
    X_train, X_test : np.ndarray of shape (n_samples, n_features)
    y_train, y_test : np.ndarray of shape (n_samples,) with integer class labels
They return: (fitted_model, metrics_dict)
"""

def _to_numpy(X: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # convert to plain numpy arrays with consistent dtypes
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)
    return X, y


def _compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float | dict[int, float]]:
    # count how many samples belong to each class
    class_ids, class_counts = np.unique(y_true, return_counts=True)

    # spotting class imbalance
    class_balance = {}
    for class_id, count in zip(class_ids, class_counts):
        class_balance[int(class_id)] = round(100 * count / len(y_true), 1)

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average='weighted'),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall': recall_score(y_true, y_pred, average='weighted'),
        'class_balance': class_balance,
    }


class MLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        activation = nn.GELU,
        hidden: tuple[int, ...] = (64, 32),
        dropout:float = 0.0,
    ):
        super().__init__()
        layers = []
        layer_input_size = input_dim

        for layer_output_size in hidden:
            # fully connected layer: maps from the previous size to this layer's size
            layers.append(nn.Linear(layer_input_size, layer_output_size))

            # add the activation function after each linear layer
            if activation == nn.GELU:
                # tanh approximation is faster and works well in practice
                layers.append(nn.GELU(approximate="tanh"))
            else:
                try:
                    # a few activations like PReLU take the layer size as an argument
                    layers.append(activation(layer_output_size))
                except TypeError:
                    # standard activations (ReLU, SELU and some others) take no arguments
                    layers.append(activation())
                    
            # we only really look into GELU, RELU and SELU, but just incase we can try others

            # dropout randomly zeros out a fraction of neurons during training,
            # which forces the network to not rely on any single neuron. This reduces overfitting
            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            layer_input_size = layer_output_size

        # final layer maps from the last hidden layer to one score per class
        layers.append(nn.Linear(layer_input_size, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_linear_classifier(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray,
) -> tuple[LogisticRegression, dict[str, float | dict[int, float]]]:
    X_train, y_train = _to_numpy(X_train, y_train)
    X_test, y_test = _to_numpy(X_test,  y_test)

    # high iteration cap so it always converges
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)

    predictions = clf.predict(X_test)
    return clf, _compute_metrics(y_test, predictions)


def train_svm(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray,
    kernel: str = 'linear',
) -> tuple[SVC, dict[str, float | dict[int, float]]]:
    X_train, y_train = _to_numpy(X_train, y_train)
    X_test, y_test = _to_numpy(X_test, y_test)

    # decision_function_shape='ovr' = one-vs-rest strategy for multi-class
    clf = SVC(kernel=kernel, decision_function_shape='ovr')
    clf.fit(X_train, y_train)

    predictions = clf.predict(X_test)
    return clf, _compute_metrics(y_test, predictions)


def train_xgboost(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test: np.ndarray,
) -> tuple[XGBClassifier, dict[str, float | dict[int, float]]]:
    X_train, y_train = _to_numpy(X_train, y_train)
    X_test, y_test = _to_numpy(X_test, y_test)

    # multi:softmax = multi-class output, mlogloss = multi-class log loss evaluation
    clf = XGBClassifier(
        objective='multi:softmax',
        eval_metric='mlogloss',
        n_estimators=30,
        max_depth=3,
        learning_rate=0.1,
        n_jobs=-1
    )
    
    clf.fit(X_train, y_train)

    predictions = clf.predict(X_test)
    return clf, _compute_metrics(y_test, predictions)


def train_mlp(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test: np.ndarray, y_test:  np.ndarray,
    hidden: tuple[int, ...]  = (64, 32),
    activation: type[nn.Module] = nn.GELU,
    dropout: float = 0.0,
    epochs: int = 200,
    lr: float = 1e-3,
    device: str | None = None,
    seed: int = 42,
) -> tuple[MLP, dict[str, float | dict[int, float]]]:
    X_train, y_train = _to_numpy(X_train, y_train)
    X_test, y_test = _to_numpy(X_test, y_test)

    # i have a GPU, but you may not. so the net will run on GPU if available, otherwise CPU. it will be slower on CPU, but should still work.
    # by default, i set the notebook to download CPU version only, it will be slow :( unfortch
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(np.unique(y_train))

    torch.manual_seed(seed)  # makes weight init and dropout reproducible

    # convert numpy arrays to PyTorch tensors and move them to the chosen device
    X_train_tensor = torch.tensor(X_train.astype(np.float32), device=device)
    y_train_tensor = torch.tensor(y_train, device=device)
    X_test_tensor = torch.tensor(X_test.astype(np.float32), device=device)
    y_test_tensor = torch.tensor(y_test, device=device)

    # DataLoader handles batching and shuffling the training data each epoch
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=256, shuffle=True)

    # build the MLP and put it on the device
    model = MLP(
        input_dim = X_train_tensor.shape[1],
        num_classes = num_classes,
        activation  = activation,
        hidden = hidden,
        dropout = dropout,
    ).to(device)

    # Adam with a small weight decay to discourage huge weights, avoid shitty models that overfit
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    # cross-entropy loss is standard for multi-class classification
    criterion = nn.CrossEntropyLoss()

    # early stopping state: track the best validation loss and save those weights
    best_val_loss = float("inf")
    best_model_weights = None
    epochs_without_improvement = 0

    c = 0
    
    for epoch in range(epochs):
        

        # ── training pass ──────────────────────────────────────────────────────
        model.train()  # training mode: enables dropout
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()                              # clear gradients from last step
            batch_loss = criterion(model(X_batch), y_batch)   # forward pass + compute loss
            batch_loss.backward()                              # backprop: compute gradients
            optimizer.step()                                   # update weights using those gradients

        # ── validation pass ────────────────────────────────────────────────────
        model.eval()  # eval mode: disables dropout so predictions are deterministic
        with torch.no_grad():
            val_loss = criterion(model(X_test_tensor), y_test_tensor).item()

        # save weights if this is the best we've seen
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_weights = model.state_dict()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            # stop early if val loss hasn't improved
            if epochs_without_improvement >= 25:
                break

    # restore the weights from the best epoch before running final predictions
    model.load_state_dict(best_model_weights)
    model.eval()

    with torch.no_grad():
        # argmax picks the class with the highest score for each sample
        test_predictions = model(X_test_tensor).argmax(dim=1).cpu().numpy()

    return model, _compute_metrics(y_test, test_predictions)

MODEL EXECUTION

In [ ]:
# load the ResNet features we extracted earlier
train_data = torch.load(r'Data\task1_data\t1_train_resnet.pt', weights_only=False)
test_data = torch.load(r'Data\task1_data\t1_test_resnet.pt',  weights_only=False)

# X = feature matrix, y = class labels (conventional ML naming)
X: np.ndarray = train_data['features']  # (3750, 2048) - all labelled images
y: np.ndarray = train_data['labels'].numpy()  # (3750,) - integer class label per image
X_test: np.ndarray = test_data['features']  # (1250, 2048) - unlabelled submission images (no y_test)

print(f'X: {X.shape}, y: {y.shape}')
print(f'X_test (submission): {X_test.shape}')
print(f'Classes: {train_data["class_to_idx"]}')

# split labelled data into train and validation sets
# stratify=y ensures both splits have the same class proportions as the full set.
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'\nX_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val: {X_val.shape}, y_val: {y_val.shape}')

In [ ]:
from sklearn.model_selection import StratifiedKFold

# ── Hyperparameters ───────────────────────────────────────────────────────────
SVM_KERNEL     = 'rbf'            # 'linear' | 'rbf' | 'poly' | 'sigmoid'

MLP_HIDDEN     = (1024, 512, 256)
MLP_ACTIVATION = nn.ReLU          # nn.GELU | nn.ReLU | nn.SELU
MLP_DROPOUT    = 0.0
MLP_EPOCHS     = 200
MLP_LR         = 1e-3
# ─────────────────────────────────────────────────────────────────────────────

# stratified k-fold: each fold has the same class distribution as the full dataset.
# we evaluate on all labelled samples to get the most reliable estimate.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def _cv_report(name: str, fold_results: list[dict]) -> None:
    """Print mean ± std across all CV folds for each metric."""
    print(f'--- {name} (5-fold CV) ---')
    
    for metric_name in ('accuracy', 'f1', 'precision', 'recall'):
        # collect this metric's value from every fold, then summarise
        fold_values = [fold[metric_name] for fold in fold_results]
        mean_value = np.mean(fold_values)
        std_value = np.std(fold_values)
        print(f'  {metric_name:<9}: {mean_value:.4f} ± {std_value:.4f}')
        
    # class balance is identical across stratified folds - print it once from fold 0
    print('  balance  :')
    for class_id, percentage in fold_results[0]['class_balance'].items():
        print(f'    class {class_id}: {percentage}%')
    print()

print('Evaluating models with 5-fold CV...\n')

# ── Linear Classifier ────────────────────────────────────────────────────────
fold_results = []
for train_idx, val_idx in cv.split(X, y):
    _, metrics = train_linear_classifier(X[train_idx], y[train_idx], X[val_idx], y[val_idx])
    fold_results.append(metrics)
_cv_report('Linear Classifier', fold_results)

# ── SVM ──────────────────────────────────────────────────────────────────────
fold_results = []
for train_idx, val_idx in cv.split(X, y):
    _, metrics = train_svm(X[train_idx], y[train_idx], X[val_idx], y[val_idx], kernel=SVM_KERNEL)
    fold_results.append(metrics)
_cv_report(f'SVM  kernel={SVM_KERNEL}', fold_results)

# ── XGBoost ──────────────────────────────────────────────────────────────────
fold_results = []
for train_idx, val_idx in cv.split(X, y):
    _, metrics = train_xgboost(X[train_idx], y[train_idx], X[val_idx], y[val_idx])
    fold_results.append(metrics)
_cv_report('XGBoost', fold_results)

# ── MLP ──────────────────────────────────────────────────────────────────────
fold_results = []
for train_idx, val_idx in cv.split(X, y):
    _, metrics = train_mlp(
        X[train_idx], y[train_idx],
        X[val_idx],   y[val_idx],
        hidden     = MLP_HIDDEN,
        activation = MLP_ACTIVATION,
        dropout    = MLP_DROPOUT,
        epochs     = MLP_EPOCHS,
        lr         = MLP_LR,
    )
    fold_results.append(metrics)
_cv_report(f'MLP  hidden={MLP_HIDDEN}  dropout={MLP_DROPOUT}', fold_results)

HYPER PARAM SELECTION

In [ ]:
from sklearn.model_selection import cross_val_score, ParameterGrid
from itertools import product

# ── SVM ───────────────────────────────────────────────────────────────────────
# per-kernel grid so each kernel only sees the params that actually affect it:
#   linear  : only C matters (C controls the margin width - lower = softer margin)
#   rbf     : C + gamma (gamma controls how far a single training point's influence reaches)
#   sigmoid : C + gamma
#   poly    : C + gamma + degree (degree is the polynomial degree)
svm_param_grid = [
    {'kernel': ['linear'],   'C': [0.01, 0.1, 1, 10, 100]},
    {'kernel': ['rbf'],      'C': [0.1, 1, 10, 100],  'gamma': ['scale', 'auto', 0.01, 0.001]},
    {'kernel': ['sigmoid'],  'C': [0.1, 1, 10, 100],  'gamma': ['scale', 'auto']},
    {'kernel': ['poly'],     'C': [0.1, 1, 10],        'gamma': ['scale', 'auto'], 'degree': [2, 3, 4]},
]
svm_configs = list(ParameterGrid(svm_param_grid))

print(f'SVM search ({len(svm_configs)} configs, 5-fold CV each):')
best_svm_f1     = -1
best_svm_params = {}

svm_bar = tqdm.tqdm(svm_configs, desc='SVM configs')
for svm_params in svm_bar:
    clf        = SVC(decision_function_shape='ovr', **svm_params)
    cv_scores  = cross_val_score(clf, X_train, y_train, cv=5, scoring='f1_weighted', n_jobs=-1)
    mean_score = cv_scores.mean()

    if mean_score > best_svm_f1:
        best_svm_f1     = mean_score
        best_svm_params = svm_params

    svm_bar.set_postfix({
        'kernel': svm_params['kernel'],
        'f1':     f'{mean_score:.4f}',
        'best':   f'{best_svm_f1:.4f}',
    })

print(f'\n  best SVM -> {best_svm_params}  f1={best_svm_f1:.4f}')

# ── MLP ───────────────────────────────────────────────────────────────────────
mlp_grid = {
    'hidden': [
        # 2 layers
        (512, 256),
        (1024, 512),
        # 3 layers
        (1024, 512, 256),
        # 4 layers
        (1024, 512, 256, 128),
        (512, 256, 128, 64),
        # 5 layers
        (1024, 512, 256, 128, 64),
        # 6 layers
        (1024, 512, 256, 128, 64, 32),
        # 7 layers
        (1024, 512, 256, 128, 64, 32, 16),
    ],
    'activation': [nn.GELU, nn.ReLU, nn.SELU],
    'dropout':    [0.0, 0.2, 0.4],
    'lr':         [1e-3, 3e-4],
}

# generate every combination of the above hyperparameters
all_configs = list(product(mlp_grid['hidden'], mlp_grid['activation'], mlp_grid['dropout'], mlp_grid['lr']))

print(f'\nMLP grid search ({len(all_configs)} configs):')
best_mlp_f1     = -1
best_mlp_config = {}

mlp_bar = tqdm.tqdm(all_configs, desc='MLP configs')
for hidden, activation, dropout, lr in mlp_bar:
    _, metrics = train_mlp(X_train, y_train, X_val, y_val,
                           hidden=hidden, activation=activation,
                           dropout=dropout, lr=lr)
    val_f1 = metrics['f1']

    if val_f1 > best_mlp_f1:
        best_mlp_f1     = val_f1
        best_mlp_config = dict(hidden=hidden, activation=activation, dropout=dropout, lr=lr)

    mlp_bar.set_postfix({
        'act':    activation.__name__,
        'drop':   dropout,
        'lr':     lr,
        'f1':     f'{val_f1:.4f}',
        'best':   f'{best_mlp_f1:.4f}',
    })

print(f'\n  best MLP -> {best_mlp_config}  f1={best_mlp_f1:.4f}')

SUBMISSION

In [ ]:
from sklearn.model_selection import train_test_split as _split_for_early_stopping

# ── Pick your model and fill in the best params from the hyper param search ──
SUBMISSION_MODEL = 'svm'   # 'linear' | 'svm' | 'xgboost' | 'mlp'

# SVM - paste best params from hyper param search
SVM_SUB_PARAMS = {'kernel': 'rbf', 'C': 10, 'gamma': 'scale'}

# MLP - paste best config from hyper param search
MLP_SUB_HIDDEN     = (512, 128)
MLP_SUB_ACTIVATION = nn.ReLU
MLP_SUB_DROPOUT    = 0.2
MLP_SUB_LR         = 3e-4
MLP_SUB_EPOCHS     = 200
# ─────────────────────────────────────────────────────────────────────────────

print(f'Training {SUBMISSION_MODEL} on full labelled set ({len(X)} samples)...')

if SUBMISSION_MODEL == 'linear':
    submission_clf = LogisticRegression(max_iter=1000)
    submission_clf.fit(np.asarray(X, dtype=np.float64), np.asarray(y, dtype=np.int64))
    predictions = submission_clf.predict(np.asarray(X_test, dtype=np.float64))

elif SUBMISSION_MODEL == 'svm':
    submission_clf = SVC(decision_function_shape='ovr', **SVM_SUB_PARAMS)
    submission_clf.fit(np.asarray(X, dtype=np.float64), np.asarray(y, dtype=np.int64))
    predictions = submission_clf.predict(np.asarray(X_test, dtype=np.float64))

elif SUBMISSION_MODEL == 'xgboost':
    submission_clf = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss')
    submission_clf.fit(np.asarray(X, dtype=np.float64), np.asarray(y, dtype=np.int64))
    predictions = submission_clf.predict(np.asarray(X_test, dtype=np.float64))

elif SUBMISSION_MODEL == 'mlp':
    # MLP needs a small val set purely for early stopping - hold out 5%.
    # we still effectively train on almost all labelled data (95%).
    X_sub_train, X_sub_val, y_sub_train, y_sub_val = _split_for_early_stopping(
        X, y, test_size=0.05, random_state=42, stratify=y
    )
    submission_model, _ = train_mlp(
        X_sub_train, y_sub_train,
        X_sub_val,   y_sub_val,
        hidden     = MLP_SUB_HIDDEN,
        activation = MLP_SUB_ACTIVATION,
        dropout    = MLP_SUB_DROPOUT,
        epochs     = MLP_SUB_EPOCHS,
        lr         = MLP_SUB_LR,
    )
    submission_model.eval()
    with torch.no_grad():
        # convert X_test to a tensor and run it through the trained model
        test_features_tensor = torch.tensor(np.asarray(X_test, dtype=np.float32))
        predictions = submission_model(test_features_tensor).argmax(dim=1).numpy()

else:
    raise ValueError(f'Unknown model: {SUBMISSION_MODEL!r}')

# extract image IDs from file paths - just the filename without the extension
image_ids = [
    os.path.splitext(os.path.basename(file_path))[0]
    for file_path in test_data['paths']
]

# build the submission dataframe and write to CSV
submission = pd.DataFrame({'image_id': image_ids, 'class_id': predictions})
submission.to_csv('submission.csv', index=False)

print(f'Saved {len(submission)} predictions -> submission.csv')

TASK 2

The general pipeline:

- Images -> ResNet Features -> PCA dim reduction -> classification

PREPROCESS IMAGES

In [ ]:
# paths to the task 2 image folders
t2_train_image_dir = r'Data\task2_data\images\train'
t2_test_image_dir  = r'Data\task2_data\images\test'

# fail early if the folders aren't where we expect them
if not os.path.exists(t2_train_image_dir):
    raise FileNotFoundError(f"Task 2 training directory not found: {t2_train_image_dir}")

if not os.path.exists(t2_test_image_dir):
    raise FileNotFoundError(f"Task 2 test directory not found: {t2_test_image_dir}")

# ── Step 1: resize + normalise and save as tensors ────────────────────────────
# reuses the same image_to_resnet function and transform from task 1
# (same ResNet preprocessing: 224x224, ImageNet normalisation)
print('Processing task 2 images...')
image_to_resnet(t2_train_image_dir, r'Data\task2_data\t2_train.pt', has_labels=True)
image_to_resnet(t2_test_image_dir,  r'Data\task2_data\t2_test.pt',  has_labels=False)

# ── Step 2: pass through ResNet50 and save 2048-d feature vectors ─────────────
print('\nExtracting ResNet features...')
extract_resnet_features(r'Data\task2_data\t2_train.pt', r'Data\task2_data\t2_train_resnet.pt')
extract_resnet_features(r'Data\task2_data\t2_test.pt',  r'Data\task2_data\t2_test_resnet.pt')

PCA DIMENSIONALITY REDUCTION

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# load task 2 ResNet features
t2_train_resnet = torch.load(r'Data\task2_data\t2_train_resnet.pt', weights_only=False)
t2_test_resnet  = torch.load(r'Data\task2_data\t2_test_resnet.pt',  weights_only=False)

X_t2       = t2_train_resnet['features']          # (417, 2048) - labelled train images
y_t2       = t2_train_resnet['labels'].numpy()    # (417,)      - class labels
X_t2_test  = t2_test_resnet['features']           # (180, 2048) - unlabelled test images

print(f'X_t2: {X_t2.shape}  |  X_t2_test: {X_t2_test.shape}')
print(f'Classes: {t2_train_resnet["class_to_idx"]}\n')

# ── Step 1: plot cumulative variance explained to pick n_components ───────────
# fit PCA with all components so we can see the full variance curve
pca_full = PCA(random_state=42)
pca_full.fit(X_t2)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# find the minimum number of components needed for 95% and 99% variance
n_50 = int(np.searchsorted(cumulative_variance, 0.50)) + 1
n_75 = int(np.searchsorted(cumulative_variance, 0.75)) + 1
n_90 = int(np.searchsorted(cumulative_variance, 0.90)) + 1
n_95 = int(np.searchsorted(cumulative_variance, 0.95)) + 1
n_99 = int(np.searchsorted(cumulative_variance, 0.99)) + 1

plt.figure(figsize=(9, 4))
plt.plot(cumulative_variance, linewidth=1.5)
plt.axhline(0.50, color='blue',   linestyle='--', label=f'50% variance ({n_50} components)')
plt.axhline(0.75, color='green',  linestyle='--', label=f'75% variance ({n_75} components)')
plt.axhline(0.90, color='purple', linestyle='--', label=f'90% variance ({n_90} components)')
plt.axhline(0.95, color='orange', linestyle='--', label=f'95% variance ({n_95} components)')
plt.axhline(0.99, color='red',    linestyle='--', label=f'99% variance ({n_99} components)')
plt.axvline(n_50 - 1, color='blue',   linestyle=':', alpha=0.6)
plt.axvline(n_75 - 1, color='green',  linestyle=':', alpha=0.6)
plt.axvline(n_90 - 1, color='purple', linestyle=':', alpha=0.6)
plt.axvline(n_95 - 1, color='orange', linestyle=':', alpha=0.6)
plt.axvline(n_99 - 1, color='red',    linestyle=':', alpha=0.6)
plt.xlabel('number of components')
plt.ylabel('cumulative variance explained')
plt.title('PCA – Task 2 ResNet Features (2048-d)')
plt.legend()
plt.tight_layout()
plt.show()

# also print a quick table for a handful of fixed sizes
print('n_components | variance explained')
print('─' * 32)
for n in [8, 16, 32, 64, 128, 256]:
    print(f'  {n:>4d}       |   {cumulative_variance[n - 1]:.3f}  ({cumulative_variance[n - 1]*100:.1f}%)')


print(f'\n  50% variance -> {n_50} components')
print(f'  75% variance -> {n_75} components')
print(f'  90% variance -> {n_90} components')
print(f'  95% variance -> {n_95} components')
print(f'  99% variance -> {n_99} components')

# ── Step 2: fit PCA with chosen n_components and transform both splits ────────
# set this to n_95, n_99, or any value from the table above
N_COMPONENTS = n_99

pca = PCA(n_components=N_COMPONENTS, random_state=42)
Z_t2_train = pca.fit_transform(X_t2)       # fit on labelled data, transform in one step
Z_t2_test  = pca.transform(X_t2_test)      # apply the same projection to test images

print(f'\nPCA: {X_t2.shape[1]}-d  ->  {Z_t2_train.shape[1]}-d')
print(f'Variance retained: {pca.explained_variance_ratio_.sum():.4f}')
print(f'Z_t2_train: {Z_t2_train.shape}')
print(f'Z_t2_test:  {Z_t2_test.shape}')

FEATURE ENGINEERING

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as _PCA

# ── Feature toggles ───────────────────────────────────────────────────────────
USE_COLOR_HISTOGRAM     = True   # 96-d  explicit colour distribution per image
USE_HOG_PCA             = True   # 100-d HOG shape/edge features (already PCA-reduced)
USE_ADDITIONAL_FEATURES = True   # 23-d  engineered handcrafted features

# ── Options ───────────────────────────────────────────────────────────────────
# Reduce colour histogram resolution by averaging every N consecutive bins.
# 96 bins total — valid factors: 1 (off), 2→48, 3→32, 4→24, 6→16, 8→12, 12→8
COLOR_HIST_REDUCE_FACTOR  = 6     # 1 = keep all 96 bins

# Apply PCA to the 23-d additional features before concatenating.
# None = keep all 23 dims (standardised);  int = reduce to that many PCA components
# Features are always standardised first so feat_1 (range ~2700) doesn't dominate
ADDITIONAL_FEATURES_PCA_N = 12  # e.g. 10
# ─────────────────────────────────────────────────────────────────────────────

def _path_to_image_id(path: str, is_train: bool) -> str:
    stem = os.path.splitext(os.path.basename(path))[0]
    return f'train_{stem}' if is_train else stem

train_ids = [_path_to_image_id(p, is_train=True)  for p in t2_train_resnet['paths']]
test_ids  = [_path_to_image_id(p, is_train=False) for p in t2_test_resnet['paths']]

def _load_and_align(csv_path: str, ids: list) -> np.ndarray:
    df = pd.read_csv(csv_path, index_col='image_id')
    return df.loc[ids].to_numpy(dtype=np.float32)

def _reduce_histogram(hist: np.ndarray, factor: int) -> np.ndarray:
    """Average every `factor` consecutive bins to lower resolution.
    factor=1 is a no-op. Requires n_bins % factor == 0."""
    if factor == 1:
        return hist
    n_bins = hist.shape[1]
    valid  = [f for f in range(1, n_bins + 1) if n_bins % f == 0]
    if n_bins % factor != 0:
        raise ValueError(f'factor {factor} does not divide n_bins ({n_bins}). '
                         f'Valid factors: {valid}')
    return hist.reshape(hist.shape[0], n_bins // factor, factor).mean(axis=2)

# PCA-reduced blocks (ResNet PCA, hog_pca) are kept as-is — their decreasing
# per-component variance is meaningful signal, not an artefact to normalise away.
# Raw features (color_histogram, additional_features) are standardised instead.
X_clf_train: np.ndarray = Z_t2_train.copy()
X_clf_test:  np.ndarray = Z_t2_test.copy()
print(f'  ResNet PCA          : {Z_t2_train.shape[1]:>4d}-d  (as-is)')

if USE_COLOR_HISTOGRAM:
    col_tr = _load_and_align(r'Data\task2_data\color_histogram.csv', train_ids)
    col_te = _load_and_align(r'Data\task2_data\color_histogram.csv', test_ids)
    col_tr = _reduce_histogram(col_tr, COLOR_HIST_REDUCE_FACTOR)
    col_te = _reduce_histogram(col_te, COLOR_HIST_REDUCE_FACTOR)
    scaler_col = StandardScaler().fit(col_tr)
    X_clf_train = np.hstack([X_clf_train, scaler_col.transform(col_tr)])
    X_clf_test  = np.hstack([X_clf_test,  scaler_col.transform(col_te)])
    res_note = f'factor={COLOR_HIST_REDUCE_FACTOR}' if COLOR_HIST_REDUCE_FACTOR > 1 else 'full res'
    print(f'+ color_histogram   : {col_tr.shape[1]:>4d}-d  (standardised, {res_note})')

if USE_HOG_PCA:
    hog_tr = _load_and_align(r'Data\task2_data\hog_pca.csv', train_ids)
    hog_te = _load_and_align(r'Data\task2_data\hog_pca.csv', test_ids)
    X_clf_train = np.hstack([X_clf_train, hog_tr])
    X_clf_test  = np.hstack([X_clf_test,  hog_te])
    print(f'+ hog_pca           : {hog_tr.shape[1]:>4d}-d  (as-is)')

if USE_ADDITIONAL_FEATURES:
    add_tr = _load_and_align(r'Data\task2_data\additional_features.csv', train_ids)
    add_te = _load_and_align(r'Data\task2_data\additional_features.csv', test_ids)
    # standardise first — feat_1 has range ~2700 vs all others ~0-1, so without
    # this feat_1 alone captures ~100% of raw variance and PCA is meaningless
    scaler_add = StandardScaler().fit(add_tr)
    add_tr_s   = scaler_add.transform(add_tr)
    add_te_s   = scaler_add.transform(add_te)
    if ADDITIONAL_FEATURES_PCA_N is not None:
        pca_add = _PCA(n_components=ADDITIONAL_FEATURES_PCA_N, random_state=42)
        add_tr  = pca_add.fit_transform(add_tr_s)
        add_te  = pca_add.transform(add_te_s)
        var_ret = pca_add.explained_variance_ratio_.sum()
        print(f'+ additional_feats  : {add_tr.shape[1]:>4d}-d  (standardised → PCA, {var_ret:.1%} var retained)')
    else:
        add_tr = add_tr_s
        add_te = add_te_s
        print(f'+ additional_feats  : {add_tr.shape[1]:>4d}-d  (standardised)')
    X_clf_train = np.hstack([X_clf_train, add_tr])
    X_clf_test  = np.hstack([X_clf_test,  add_te])

print(f'{"─"*38}')
print(f'  Final               : {X_clf_train.shape[1]:>4d}-d  ({X_clf_train.shape[0]} train, {X_clf_test.shape[0]} test)')

CLASSIFICATION MODELS

In [ ]:
"""
Task 2 – additional classifiers.
We reuse train_linear_classifier, train_svm, train_xgboost and train_mlp from Task 1.
Two new models suited to 150-300-d PCA features with ~417 samples:

  kNN          – simple distance-based classifier; works well in lower dims after PCA.
                 k=10 balances noise sensitivity vs. over-smoothing at this dataset size.

  Random Forest – ensemble of decorrelated trees; handles high-dim features well,
                  robust to small datasets, and built-in feature importance as a bonus.

Note on kMeans: kMeans is an unsupervised clustering algorithm — it has no concept of
class labels during training, so standard accuracy / F1 metrics don't apply directly.
Random Forest is the recommended supervised replacement here.
"""

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier


def train_knn(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test:  np.ndarray, y_test:  np.ndarray,
    n_neighbors: int = 10,
) -> tuple[KNeighborsClassifier, dict]:
    X_train, y_train = _to_numpy(X_train, y_train)
    X_test,  y_test  = _to_numpy(X_test,  y_test)
    clf = KNeighborsClassifier(n_neighbors=n_neighbors)
    clf.fit(X_train, y_train)
    return clf, _compute_metrics(y_test, clf.predict(X_test))


def train_random_forest(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test:  np.ndarray, y_test:  np.ndarray,
    n_estimators: int = 50,
    max_features: str = 'sqrt',   # sqrt(n_features) per split — standard RF default
) -> tuple[RandomForestClassifier, dict]:
    X_train, y_train = _to_numpy(X_train, y_train)
    X_test,  y_test  = _to_numpy(X_test,  y_test)
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_features=max_features,
        random_state=42,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    return clf, _compute_metrics(y_test, clf.predict(X_test))

MODEL EXECUTION

In [ ]:
from sklearn.model_selection import StratifiedKFold

# ── Hyperparameters ───────────────────────────────────────────────────────────
T2_SVM_KERNEL    = 'rbf'   # 'linear' | 'rbf' | 'poly' | 'sigmoid'
T2_KNN_K         = 15      # try 5–15; larger k = smoother decision boundary
T2_RF_ESTIMATORS = 50     # more trees = more stable, diminishing returns past ~300

T2_MLP_HIDDEN    = (256, 128)  # smaller layers than Task 1 — only 417 training samples
T2_MLP_ACT       = nn.ReLU
T2_MLP_DROPOUT   = 0.1
T2_MLP_EPOCHS    = 300
T2_MLP_LR        = 1e-3
# ─────────────────────────────────────────────────────────────────────────────

cv_t2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Evaluating on Task 2 features  '
      f'({X_clf_train.shape[1]}-d, {len(y_t2)} samples, 5-fold CV)...\n')

# ── Logistic Regression ───────────────────────────────────────────────────────
fold_results = []
for tr, va in cv_t2.split(X_clf_train, y_t2):
    _, m = train_linear_classifier(X_clf_train[tr], y_t2[tr], X_clf_train[va], y_t2[va])
    fold_results.append(m)
_cv_report('Logistic Regression', fold_results)

# ── SVM ───────────────────────────────────────────────────────────────────────
fold_results = []
for tr, va in cv_t2.split(X_clf_train, y_t2):
    _, m = train_svm(X_clf_train[tr], y_t2[tr], X_clf_train[va], y_t2[va], kernel=T2_SVM_KERNEL)
    fold_results.append(m)
_cv_report(f'SVM  kernel={T2_SVM_KERNEL}', fold_results)

# ── kNN ───────────────────────────────────────────────────────────────────────
fold_results = []
for tr, va in cv_t2.split(X_clf_train, y_t2):
    _, m = train_knn(X_clf_train[tr], y_t2[tr], X_clf_train[va], y_t2[va], n_neighbors=T2_KNN_K)
    fold_results.append(m)
_cv_report(f'kNN  k={T2_KNN_K}', fold_results)

# ── Random Forest ─────────────────────────────────────────────────────────────
fold_results = []
for tr, va in cv_t2.split(X_clf_train, y_t2):
    _, m = train_random_forest(X_clf_train[tr], y_t2[tr], X_clf_train[va], y_t2[va],
                               n_estimators=T2_RF_ESTIMATORS)
    fold_results.append(m)
_cv_report(f'Random Forest  n={T2_RF_ESTIMATORS}', fold_results)

# ── XGBoost ───────────────────────────────────────────────────────────────────
fold_results = []
for tr, va in cv_t2.split(X_clf_train, y_t2):
    _, m = train_xgboost(X_clf_train[tr], y_t2[tr], X_clf_train[va], y_t2[va])
    fold_results.append(m)
_cv_report('XGBoost', fold_results)

# ── MLP ───────────────────────────────────────────────────────────────────────
fold_results = []
for tr, va in cv_t2.split(X_clf_train, y_t2):
    _, m = train_mlp(
        X_clf_train[tr], y_t2[tr], X_clf_train[va], y_t2[va],
        hidden=T2_MLP_HIDDEN, activation=T2_MLP_ACT,
        dropout=T2_MLP_DROPOUT, epochs=T2_MLP_EPOCHS, lr=T2_MLP_LR,
    )
    fold_results.append(m)
_cv_report(f'MLP  hidden={T2_MLP_HIDDEN}  dropout={T2_MLP_DROPOUT}', fold_results)

HYPER PARAM SEARCH

In [21]:
from itertools import product as _product
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# =============================================================================
#  TASK 2 - JOINT HYPER-PARAMETER SEARCH
#
#  Searches over feature engineering params AND model params simultaneously.
#  Uses a single stratified 80/20 train/val split -- fast but noisier than CV.
#  Results are ranked by weighted F1 on the val set.
#
#  Models run in three passes so fast models finish across ALL feature configs
#  before slower ones start.  Order:
#    Pass 1 (fast)   -- LogReg, SVM
#    Pass 2 (medium) -- RF, XGBoost
#    Pass 3 (slow)   -- MLP
#
#  Estimated runtime (CPU):  ~2-4 min
# =============================================================================

# -- Feature engineering search grids -----------------------------------------
T2_SEARCH_RESNET_PCA  = [n_50,n_75, n_90, n_95, n_99]
T2_SEARCH_HIST_FACTOR = [1, 2, 6, 12]
T2_SEARCH_ADD_PCA     = [None, 5, 10, 15]

# -- Model search grids --------------------------------------------------------
T2_SEARCH_LOGREG = [
    {'C': 0.1},
    {'C': 1},
    {'C': 10},
]

T2_SEARCH_SVM = [
    {'kernel': 'rbf',    'C': 1,   'gamma': 'scale'},
    {'kernel': 'rbf',    'C': 10,  'gamma': 'scale'},
    {'kernel': 'rbf',    'C': 100, 'gamma': 'scale'},
]

T2_SEARCH_RF = [
    {'n_estimators': 50, 'max_features': 'log2'},
]

T2_SEARCH_XGB = [
    {'n_estimators': 50, 'max_depth': 6,  'learning_rate': 0.1},
    {'n_estimators': 50, 'max_depth': 4,  'learning_rate': 0.1},
]

T3_SEARCH_MLP = [
    {'hidden': (256, 128), 'activation': nn.ReLU, 'dropout': 0.1, 'lr': 1e-3, 'epochs': 300},
    {'hidden': (256, 128), 'activation': nn.GELU, 'dropout': 0.1, 'lr': 1e-3, 'epochs': 300},
]

# -----------------------------------------------------------------------------

# -- Pre-load raw CSVs once ---------------------------------------------------
_s_col_tr = _load_and_align(r'Data\task2_data\color_histogram.csv',     train_ids)
_s_hog_tr = _load_and_align(r'Data\task2_data\hog_pca.csv',             train_ids)
_s_add_tr = _load_and_align(r'Data\task2_data\additional_features.csv', train_ids)

# -- Pre-compute ResNet PCA projections ---------------------------------------
print('Pre-computing ResNet PCA projections...')
_resnet_cache: dict[int, np.ndarray] = {}
for _nc in T2_SEARCH_RESNET_PCA:
    _p = PCA(n_components=_nc, random_state=42)
    _resnet_cache[_nc] = _p.fit_transform(X_t2).astype(np.float64)
    print(f'  n_components={_nc}: {_resnet_cache[_nc].shape[1]}-d')

# -- Single stratified 80/20 split -------------------------------------------
_tr_idx, _va_idx = next(
    StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_t2, y_t2)
)
print(f'\nTrain split: {len(_tr_idx)} samples  |  Val split: {len(_va_idx)} samples')

def _build_search_features(n_comp: int, hist_factor: int, add_pca_n) -> np.ndarray:
    # Returns (417, total_dims) -- scalers/PCA fit on train split only
    Xtr = _resnet_cache[n_comp]

    col    = _reduce_histogram(_s_col_tr.copy(), hist_factor)
    sc_col = StandardScaler().fit(col[_tr_idx])
    Xtr    = np.hstack([Xtr, sc_col.transform(col)])

    Xtr = np.hstack([Xtr, _s_hog_tr.astype(np.float64)])

    sc_add = StandardScaler().fit(_s_add_tr[_tr_idx])
    add_s  = sc_add.transform(_s_add_tr)
    if add_pca_n is not None:
        pa    = _PCA(n_components=add_pca_n, random_state=42)
        pa.fit(add_s[_tr_idx])
        add_s = pa.transform(add_s)
    Xtr = np.hstack([Xtr, add_s])
    return Xtr

# -- Pre-build and cache all feature matrices ---------------------------------
# Caching avoids rebuilding the same matrix in each of the three passes.
# 27 configs x ~400d x 417 samples x 8 bytes ~ 35 MB total -- fine.
print('\nPre-building feature matrices...')
_feat_configs  = list(_product(T2_SEARCH_RESNET_PCA, T2_SEARCH_HIST_FACTOR, T2_SEARCH_ADD_PCA))
_feat_matrices = {}
for _key in _feat_configs:
    _feat_matrices[_key] = _build_search_features(*_key)
print(f'  {len(_feat_configs)} feature configs ready')

# -- Search setup -------------------------------------------------------------
_n_model_cfgs = (len(T2_SEARCH_LOGREG) + len(T2_SEARCH_SVM) + len(T2_SEARCH_RF)
                 + len(T2_SEARCH_XGB) + len(T3_SEARCH_MLP))
_total = len(_feat_configs) * _n_model_cfgs
print(f'\n{len(_feat_configs)} feature configs x {_n_model_cfgs} model configs = {_total} evaluations\n')

_search_results: list[dict] = []
_pbar = tqdm.tqdm(total=_total, desc='Search')

def _record_result(model, params, n_comp, hist_f, add_n, dims, f1_val):
    _search_results.append({
        'model':       model,
        'params':      params,
        'resnet_pca':  n_comp,
        'hist_factor': hist_f,
        'add_pca':     str(add_n),
        'dims':        dims,
        'f1':          float(f1_val),
    })

def _get_splits(key):
    _X = _feat_matrices[key]
    return _X[_tr_idx], _X[_va_idx], y_t2[_tr_idx], y_t2[_va_idx], _X.shape[1]

# =============================================================================
#  PASS 1 -- FAST  (LogReg, SVM)
# =============================================================================
_pbar.set_description('Pass 1/3 -- fast (LogReg, SVM)')
for _nc, _hf, _an in _feat_configs:
    _Xtr_s, _Xva_s, _ytr_s, _yva_s, _d = _get_splits((_nc, _hf, _an))

    for p in T2_SEARCH_LOGREG:
        clf = LogisticRegression(max_iter=2000, **p)
        clf.fit(_Xtr_s, _ytr_s)
        _record_result('LogReg', f"C={p['C']}", _nc, _hf, _an, _d,
                       f1_score(_yva_s, clf.predict(_Xva_s), average='weighted'))
        _pbar.update(1)

    for p in T2_SEARCH_SVM:
        clf = SVC(decision_function_shape='ovr', **p)
        clf.fit(_Xtr_s, _ytr_s)
        g = f" gamma={p['gamma']}" if 'gamma' in p else ''
        _record_result('SVM', f"kernel={p['kernel']} C={p['C']}{g}", _nc, _hf, _an, _d,
                       f1_score(_yva_s, clf.predict(_Xva_s), average='weighted'))
        _pbar.update(1)

# =============================================================================
#  PASS 2 -- MEDIUM  (RF, XGBoost)
# =============================================================================
_pbar.set_description('Pass 2/3 -- medium (RF, XGBoost)')
for _nc, _hf, _an in _feat_configs:
    _Xtr_s, _Xva_s, _ytr_s, _yva_s, _d = _get_splits((_nc, _hf, _an))

    for p in T2_SEARCH_RF:
        clf = RandomForestClassifier(random_state=42, n_jobs=-1, **p)
        clf.fit(_Xtr_s, _ytr_s)
        _record_result('RF', f"n={p['n_estimators']} feat={p['max_features']}", _nc, _hf, _an, _d,
                       f1_score(_yva_s, clf.predict(_Xva_s), average='weighted'))
        _pbar.update(1)

    for p in T2_SEARCH_XGB:
        clf = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', verbosity=0, **p)
        clf.fit(_Xtr_s, _ytr_s)
        _record_result('XGBoost', f"n={p['n_estimators']} depth={p['max_depth']} lr={p['learning_rate']}",
                       _nc, _hf, _an, _d,
                       f1_score(_yva_s, clf.predict(_Xva_s), average='weighted'))
        _pbar.update(1)

# =============================================================================
#  PASS 3 -- SLOW  (MLP)
# =============================================================================
_pbar.set_description('Pass 3/3 -- slow (MLP)')
for _nc, _hf, _an in _feat_configs:
    _Xtr_s, _Xva_s, _ytr_s, _yva_s, _d = _get_splits((_nc, _hf, _an))

    for p in T3_SEARCH_MLP:
        model, metrics = train_mlp(
            _Xtr_s, _ytr_s, _Xva_s, _yva_s,
            hidden=p['hidden'], activation=p['activation'],
            dropout=p['dropout'], epochs=p['epochs'], lr=p['lr'],
        )
        _record_result('MLP', f"hidden={p['hidden']} act={p['activation'].__name__} drop={p['dropout']} lr={p['lr']}",
                       _nc, _hf, _an, _d, metrics['f1'])
        _pbar.update(1)


_pbar.close()

# -- Display top 10 per model -------------------------------------------------
_res_df = (pd.DataFrame(_search_results)
             .sort_values(['model', 'f1'], ascending=[True, False])
             .reset_index(drop=True))

_W = 108

for _model_name, _model_df in _res_df.groupby("model"):

    print(f'\n{"="*_W}')
    print(f'  TOP 10 {_model_name} configs  --  ranked by weighted F1')
    print(f'{"="*_W}')
    print(f'  {"#":<3}  {"Model":<8}  {"F1":<8}  {"Model params":<38}  {"PCA":<6}  {"Hist":<5}  {"AddPCA":<7}  Dims')
    print(f'  {"-"*(_W-2)}')

    for _rank, (_, _row) in enumerate(_model_df.head(10).iterrows(), start=1):
        print(
            f'  {_rank:<3}  '
            f'{_row["model"]:<8}  '
            f'{_row["f1"]:.4f}    '
            f'{str(_row["params"]):<38}  '
            f'{_row["resnet_pca"]:<6}  '
            f'{_row["hist_factor"]:<5}  '
            f'{_row["add_pca"]:<7}  '
            f'{_row["dims"]}'
        )

    print(f'{"="*_W}')

print('\nFull results available in _res_df')

Pre-computing ResNet PCA projections...
  n_components=9: 9-d
  n_components=42: 42-d
  n_components=115: 115-d
  n_components=177: 177-d
  n_components=303: 303-d

Train split: 333 samples  |  Val split: 84 samples

Pre-building feature matrices...
  80 feature configs ready

80 feature configs x 11 model configs = 880 evaluations



Pass 3/3 -- slow (MLP): 100%|██████████| 880/880 [03:07<00:00,  4.68it/s]          


  TOP 10 LogReg configs  --  ranked by weighted F1
  #    Model     F1        Model params                            PCA     Hist   AddPCA   Dims
  ----------------------------------------------------------------------------------------------------------
  1    LogReg    0.9044    C=1                                     115     12     5        228
  2    LogReg    0.9024    C=1                                     177     12     None     308
  3    LogReg    0.9024    C=1                                     177     12     5        290
  4    LogReg    0.9024    C=1                                     177     12     10       295
  5    LogReg    0.9024    C=10                                    177     12     10       295
  6    LogReg    0.9024    C=1                                     177     12     15       300
  7    LogReg    0.8925    C=1                                     115     12     10       233
  8    LogReg    0.8925    C=10                                    115     12 

SUBMISSION

In [23]:
from sklearn.model_selection import train_test_split as _t2_split

# ── Pick your model ───────────────────────────────────────────────────────────
T2_SUBMISSION_MODEL = 'linear'   # 'linear' | 'svm' | 'knn' | 'rf' | 'xgboost' | 'mlp'

# SVM - paste best params from hyper param search above (or leave default)
T2_SVM_SUB_PARAMS = {'kernel': 'rbf', 'C': 1, 'gamma': 'scale'}

# kNN
T2_KNN_SUB_K = 10

# Random Forest
T2_RF_SUB_ESTIMATORS = 300

# MLP
T2_MLP_SUB_HIDDEN     = (256, 128)
T2_MLP_SUB_ACTIVATION = nn.ReLU
T2_MLP_SUB_DROPOUT    = 0.2
T2_MLP_SUB_LR         = 1e-3
T2_MLP_SUB_EPOCHS     = 300
# ─────────────────────────────────────────────────────────────────────────────

# X_clf_train / X_clf_test are the combined + standardised feature matrices
# built by the FEATURE ENGINEERING cell above — re-run that cell if you change the toggles
print(f'Training {T2_SUBMISSION_MODEL} on full Task 2 labelled set '
      f'({X_clf_train.shape[0]} samples, {X_clf_train.shape[1]}-d)...')

X_tr = np.asarray(X_clf_train, dtype=np.float64)
X_te = np.asarray(X_clf_test,  dtype=np.float64)
y_np = np.asarray(y_t2,        dtype=np.int64)

if T2_SUBMISSION_MODEL == 'linear':
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_tr, y_np)
    t2_predictions = clf.predict(X_te)

elif T2_SUBMISSION_MODEL == 'svm':
    clf = SVC(decision_function_shape='ovr', **T2_SVM_SUB_PARAMS)
    clf.fit(X_tr, y_np)
    t2_predictions = clf.predict(X_te)

elif T2_SUBMISSION_MODEL == 'knn':
    clf = KNeighborsClassifier(n_neighbors=T2_KNN_SUB_K)
    clf.fit(X_tr, y_np)
    t2_predictions = clf.predict(X_te)

elif T2_SUBMISSION_MODEL == 'rf':
    clf = RandomForestClassifier(
        n_estimators=T2_RF_SUB_ESTIMATORS, max_features='sqrt',
        random_state=42, n_jobs=-1,
    )
    clf.fit(X_tr, y_np)
    t2_predictions = clf.predict(X_te)

elif T2_SUBMISSION_MODEL == 'xgboost':
    clf = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss')
    clf.fit(X_tr, y_np)
    t2_predictions = clf.predict(X_te)

elif T2_SUBMISSION_MODEL == 'mlp':
    # MLP needs a small val split for early stopping only
    X_sub_tr, X_sub_val, y_sub_tr, y_sub_val = _t2_split(
        X_clf_train, y_t2, test_size=0.05, random_state=42, stratify=y_t2
    )
    t2_model, _ = train_mlp(
        X_sub_tr, y_sub_tr, X_sub_val, y_sub_val,
        hidden=T2_MLP_SUB_HIDDEN, activation=T2_MLP_SUB_ACTIVATION,
        dropout=T2_MLP_SUB_DROPOUT, epochs=T2_MLP_SUB_EPOCHS, lr=T2_MLP_SUB_LR,
    )
    t2_model.eval()
    with torch.no_grad():
        t2_predictions = t2_model(
            torch.tensor(np.asarray(X_clf_test, dtype=np.float32))
        ).argmax(dim=1).numpy()

else:
    raise ValueError(f'Unknown model: {T2_SUBMISSION_MODEL!r}')

# extract image IDs from file paths
t2_image_ids = [
    os.path.splitext(os.path.basename(p))[0]
    for p in t2_test_resnet['paths']
]

t2_submission = pd.DataFrame({'image_id': t2_image_ids, 'class_id': t2_predictions})
t2_submission.to_csv('t2_submission.csv', index=False)
print(f'Saved {len(t2_submission)} predictions -> t2_submission.csv')

Training linear on full Task 2 labelled set (417 samples, 431-d)...
Saved 180 predictions -> t2_submission.csv
